# 5G RCF Anomaly Detection Demo

This notebook demonstrates **AWS Managed Prometheus RCF anomaly detection** on a live 5G network.

**Scenario**: A bad config push to AMF1 causes ~50 users to lose registration. RCF detects the anomaly instantly. The DevOps Agent correlates with infrastructure metrics to identify root cause.

---

## Architecture

![Architecture](architecture.png)

## RCF Data Flow

![RCF Dataflow](rcf-dataflow.png)

## Demo Scenario (Before → Fault → Recovery)

![Fault Scenario](fault-scenario.png)

---

## Setup

In [ ]:
import boto3
import json
import time
import subprocess
import os
from datetime import datetime, timezone, timedelta
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
import requests

# Configuration
REGION = 'us-east-1'
WORKSPACE_ID = 'ws-185ff7f8-c698-4d0e-9135-945b03aeccd1'
AMP_QUERY_URL = f'https://aps-workspaces.{REGION}.amazonaws.com/workspaces/{WORKSPACE_ID}/api/v1/query'
EKS_CLUSTER = 'open5gs-amp-cluster'

session = boto3.Session(region_name=REGION)
credentials = session.get_credentials().get_frozen_credentials()

def query_amp(promql):
    """Execute a PromQL query against AMP with SigV4 auth."""
    params = {'query': promql}
    req = AWSRequest(method='POST', url=AMP_QUERY_URL, data=params)
    SigV4Auth(credentials, 'aps', REGION).add_auth(req)
    resp = requests.post(AMP_QUERY_URL, data=params, headers=dict(req.headers))
    return resp.json()['data']['result']

# Install kubectl if not present
if not os.path.exists('/usr/local/bin/kubectl'):
    print('Installing kubectl...')
    subprocess.run(['curl', '-sLO', 'https://dl.k8s.io/release/v1.31.0/bin/linux/amd64/kubectl'], capture_output=True)
    subprocess.run(['chmod', '+x', 'kubectl'], capture_output=True)
    subprocess.run(['sudo', 'mv', 'kubectl', '/usr/local/bin/'], capture_output=True)

# Setup kubeconfig
subprocess.run(['aws', 'eks', 'update-kubeconfig', '--region', REGION, '--name', EKS_CLUSTER],
               capture_output=True)

print(f'✓ Connected to AMP workspace: {WORKSPACE_ID}')
print(f'  Region: {REGION}')
print(f'  EKS: {EKS_CLUSTER}')
print(f'  kubectl: {subprocess.run(["kubectl", "version", "--client", "--short"], capture_output=True, text=True).stdout.strip()}')

## Step 1: Verify Baseline (Healthy State)

All 100 UEs should be registered across 2 AMFs. RCF score should be 0.

In [ ]:
# Check registered subscribers per AMF
results = query_amp('fivegs_amffunction_rm_registeredsubnbr')
print('═══ Registered Subscribers (per AMF) ═══')
total = 0
for r in results:
    pod = r['metric'].get('pod', 'unknown')
    value = int(r['value'][1])
    total += value
    print(f'  {pod}: {value} UEs')
print(f'  ─────────────────────')
print(f'  TOTAL: {total} UEs registered')
print()

# Check RCF detector state
rcf_metrics = query_amp('{__name__=~"anomaly_detector:.+", alias="5g-registered-subscribers"}')
print('═══ RCF Anomaly Detector ═══')
for r in rcf_metrics:
    name = r['metric']['__name__'].replace('anomaly_detector:', '')
    value = r['value'][1]
    print(f'  {name:12s}: {value}')
print()
print('✓ Baseline healthy' if total >= 80 else '⚠ Subscribers below expected')

## Step 2: Inject Fault (Bad Config Push to AMF1)

This simulates a **human error**: pushing a broken configuration to AMF1.

- The config is missing the required `time.t3512` field
- AMF1 reads it, fails validation, and crashes immediately
- Kubernetes restarts it → crashes again → **CrashLoopBackOff**
- ~50 UEs on AMF1 (TAC=1) lose their registration
- AMF2 (TAC=2) remains completely unaffected

In [ ]:
# Push broken config to AMF1 (missing required time.t3512 field)
broken_config = '''sbi:\n  server:\n    no_tls: true\n  client:\n    no_tls: true\namf:\n  sbi:\n    - addr: 0.0.0.0\n      port: 7777\n  ngap:\n    - addr: 0.0.0.0\n  metrics:\n    - addr: 0.0.0.0\n      port: 9090\n  guami:\n    - plmn_id: {mcc: 999, mnc: 70}\n      amf_id: {region: 2, set: 1}\n  tai:\n    - plmn_id: {mcc: 999, mnc: 70}\n      tac: 1\n  plmn_support:\n    - plmn_id: {mcc: 999, mnc: 70}\n      s_nssai:\n        - sst: 1\n  security:\n    integrity_order: [NIA2, NIA1, NIA0]\n    ciphering_order: [NEA0, NEA1, NEA2]\n  network_name:\n    full: Open5GS\n  amf_name: open5gs-amf1\nscp:\n  sbi:\n    - addr: scp.open5gs.svc.cluster.local\n      port: 7777\n'''

print('═══ FAULT INJECTION ═══')
print('Pushing broken config to AMF1 (missing time.t3512)...')
print()

# Create broken configmap and apply
result = subprocess.run(
    ['kubectl', 'create', 'configmap', 'amf1-config', '-n', 'open5gs',
     f'--from-literal=amf.yaml={broken_config}',
     '--dry-run=client', '-o', 'yaml'],
    capture_output=True, text=True)
subprocess.run(['kubectl', 'apply', '-f', '-'], input=result.stdout, capture_output=True, text=True)

# Restart AMF1
subprocess.run(['kubectl', 'rollout', 'restart', 'deploy/amf1', '-n', 'open5gs'], capture_output=True)

print('✗ Bad config pushed. AMF1 will enter CrashLoopBackOff.')
print('  Impact: ~50 UEs will lose registration within 30s.')
print()
print('Waiting 45s for impact to propagate to AMP...')
time.sleep(45)

## Step 3: Observe the Anomaly

RCF evaluates every 30s. After the drop, the score should spike above 0.1.

In [ ]:
# Check the impact
results = query_amp('fivegs_amffunction_rm_registeredsubnbr')
print('═══ AFTER FAULT: Registered Subscribers ═══')
total = 0
for r in results:
    pod = r['metric'].get('pod', 'unknown')
    value = int(r['value'][1])
    total += value
    status = '✗ DOWN' if 'amf1' in pod and value == 0 else '✓ OK'
    print(f'  {pod}: {value} UEs  {status}')
print(f'  ─────────────────────')
print(f'  TOTAL: {total} UEs (was 100)')
print(f'  IMPACT: {100 - total} users lost service')
print()

# Check RCF score
rcf_metrics = query_amp('{__name__=~"anomaly_detector:.+", alias="5g-registered-subscribers"}')
print('═══ RCF Anomaly Detector ═══')
for r in rcf_metrics:
    name = r['metric']['__name__'].replace('anomaly_detector:', '')
    value = r['value'][1]
    indicator = ' ← ANOMALY DETECTED!' if name == 'score' and float(value) > 0 else ''
    print(f'  {name:12s}: {value}{indicator}')

## Step 4: Root Cause Analysis

The DevOps Agent correlates the 5G anomaly with infrastructure metrics:
1. Which AMF is affected? (per-AMF breakdown)
2. Is it crashing? (pod restart count)
3. Is it a node issue? (node schedulability)
4. Conclusion: config error on AMF1

In [ ]:
print('═══ ROOT CAUSE ANALYSIS ═══')
print()

# 1. Which AMF is affected?
print('1. Per-AMF breakdown:')
results = query_amp('fivegs_amffunction_rm_registeredsubnbr')
for r in results:
    pod = r['metric'].get('pod', '')
    value = r['value'][1]
    print(f'   {pod}: {value} registered', '← AFFECTED' if int(value) == 0 else '')
print()

# 2. Pod restarts
print('2. Pod restart count:')
results = query_amp('kube_pod_container_status_restarts_total{namespace="open5gs", pod=~"amf.*"}')
for r in results:
    pod = r['metric'].get('pod', '')
    restarts = r['value'][1]
    print(f'   {pod}: {restarts} restarts', '← CRASH LOOP!' if int(float(restarts)) > 2 else '')
print()

# 3. Node health
print('3. Node health:')
results = query_amp('kube_node_spec_unschedulable')
if not results:
    print('   All nodes schedulable ✓ (not a node issue)')
else:
    for r in results:
        print(f'   {r["metric"].get("node")}: unschedulable={r["value"][1]}')
print()

# Conclusion
print('═══ CONCLUSION ═══')
print('Root cause: AMF1 is in CrashLoopBackOff after a config change.')
print('Impact: ~50 users on TAC=1 lost registration.')
print('AMF2 (TAC=2) is healthy — not a cluster-wide issue.')
print('Recommendation: Rollback amf1-config ConfigMap.')

## Step 5: Recovery

Restore the valid configuration to AMF1. UEs will auto-re-register.

In [ ]:
valid_config = '''sbi:\n  server:\n    no_tls: true\n  client:\n    no_tls: true\n\ntime:\n  t3512:\n    value: 540\n\namf:\n  sbi:\n    - addr: 0.0.0.0\n      port: 7777\n  ngap:\n    - addr: 0.0.0.0\n  metrics:\n    - addr: 0.0.0.0\n      port: 9090\n  guami:\n    - plmn_id: {mcc: 999, mnc: 70}\n      amf_id: {region: 2, set: 1}\n  tai:\n    - plmn_id: {mcc: 999, mnc: 70}\n      tac: 1\n  plmn_support:\n    - plmn_id: {mcc: 999, mnc: 70}\n      s_nssai:\n        - sst: 1\n  security:\n    integrity_order: [NIA2, NIA1, NIA0]\n    ciphering_order: [NEA0, NEA1, NEA2]\n  network_name:\n    full: Open5GS\n  amf_name: open5gs-amf1\nscp:\n  sbi:\n    - addr: scp.open5gs.svc.cluster.local\n      port: 7777\n'''

print('═══ RECOVERY ═══')
print('Restoring valid config to AMF1...')

result = subprocess.run(
    ['kubectl', 'create', 'configmap', 'amf1-config', '-n', 'open5gs',
     f'--from-literal=amf.yaml={valid_config}',
     '--dry-run=client', '-o', 'yaml'],
    capture_output=True, text=True)
subprocess.run(['kubectl', 'apply', '-f', '-'], input=result.stdout, capture_output=True, text=True)
subprocess.run(['kubectl', 'rollout', 'restart', 'deploy/amf1', '-n', 'open5gs'], capture_output=True)

print('✓ Valid config restored. Waiting 60s for UEs to re-register...')
time.sleep(60)

# Verify recovery
results = query_amp('sum(fivegs_amffunction_rm_registeredsubnbr)')
total = int(results[0]['value'][1]) if results else 0
print(f'\n═══ POST-RECOVERY ═══')
print(f'  Registered subscribers: {total}')
print(f'  Status: {"✓ RECOVERED" if total >= 80 else "⏳ Still recovering..."}')

---

## Summary

| Phase | registeredsubnbr | RCF Score | Root Cause |
|---|---|---|---|
| Healthy | 100 | 0 | — |
| Fault injected | ~50 | >0.1 | AMF1 CrashLoopBackOff (bad config) |
| Recovered | 100 | 0 | Config rolled back |

### Key Takeaways
1. **RCF detects onset** — the moment subscribers drop, score spikes
2. **Infra correlation** — pod restarts + per-AMF breakdown pinpoints root cause
3. **Partial impact** — only TAC=1 users affected; TAC=2 is healthy
4. **Fast recovery** — fix config, restart, UEs auto-re-register